In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA gold;

In [0]:
CREATE TABLE IF NOT EXISTS gold.characters_behavior_daily (
  snapshot_date DATE,
  character_id STRING,
  world_id STRING,
  account_status STRING,
  sex STRING,
  vocation STRING,
  level INT,
  level_delta_1d INT,
  level_delta_7d INT,
  level_delta_30d INT,
  level_delta_90d INT,
  first_seen_date DATE,
  last_login_date DATE,
  days_since_last_login INT,
  is_active_1d BOOLEAN,
  is_active_7d BOOLEAN,
  is_active_30d BOOLEAN,
  is_active_90d BOOLEAN,
  lifecycle_stage STRING,
  lifecycle_event STRING,
  in_guild BOOLEAN,
  owns_house BOOLEAN,
  is_married BOOLEAN,
  loyalty_title STRING,
  achievement_points INT,
  processed_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (snapshot_date);

In [0]:
COMMENT ON TABLE gold.characters_behavior_daily IS 'Daily behavioral snapshot of Tibia characters used for longitudinal community health analysis.';
COMMENT ON COLUMN gold.characters_behavior_daily.snapshot_date IS 'Reference date of the data snapshot, representing the state of the source data at extraction time.';
COMMENT ON COLUMN gold.characters_behavior_daily.character_id IS 'Stable unique identifier of the character.';
COMMENT ON COLUMN gold.characters_behavior_daily.world_id IS 'Stable unique identifier of the Tibia world.';
COMMENT ON COLUMN gold.characters_behavior_daily.account_status IS 'Indicates whether the character account is Free or Premium.';
COMMENT ON COLUMN gold.characters_behavior_daily.sex IS 'Current sex of the character on the snapshot date.';
COMMENT ON COLUMN gold.characters_behavior_daily.vocation IS 'Current vocation of the character on the snapshot date.';
COMMENT ON COLUMN gold.characters_behavior_daily.level IS 'Current level of the character on the snapshot date.';
COMMENT ON COLUMN gold.characters_behavior_daily.level_delta_1d IS 'Difference in character level compared to the previous day. Formula: level(D) - level(D-1).';
COMMENT ON COLUMN gold.characters_behavior_daily.level_delta_7d IS 'Difference in character level compared to 7 days ago. Formula: level(D) - level(D-7).';
COMMENT ON COLUMN gold.characters_behavior_daily.level_delta_30d IS 'Difference in character level compared to 30 days ago. Formula: level(D) - level(D-30).';
COMMENT ON COLUMN gold.characters_behavior_daily.level_delta_90d IS 'Difference in character level compared to 90 days ago. Formula: level(D) - level(D-90).';
COMMENT ON COLUMN gold.characters_behavior_daily.first_seen_date IS 'Earliest date on which the character was observed in the dataset based on ingestion history, not necessarily the character creation date.';
COMMENT ON COLUMN gold.characters_behavior_daily.last_login_date IS 'Most recent known login date of the character as reported by the source API at the time of ingestion.';
COMMENT ON COLUMN gold.characters_behavior_daily.days_since_last_login IS 'Number of days between the snapshot_date and the character last_login_date, representing inactivity duration. Formula: snapshot_date - last_login_date.';
COMMENT ON COLUMN gold.characters_behavior_daily.is_active_1d IS 'Indicates whether the character was active within the last 1 day relative to snapshot_date.';
COMMENT ON COLUMN gold.characters_behavior_daily.is_active_7d IS 'Indicates whether the character was active within the last 7 days relative to snapshot_date.';
COMMENT ON COLUMN gold.characters_behavior_daily.is_active_30d IS 'Indicates whether the character was active within the last 30 days relative to snapshot_date.';
COMMENT ON COLUMN gold.characters_behavior_daily.is_active_90d IS 'Indicates whether the character was active within the last 90 days relative to snapshot_date.';
COMMENT ON COLUMN gold.characters_behavior_daily.lifecycle_stage IS 'Player lifecycle classification on the snapshot date. Mutually exclusive categories: new, returning, active, dormant, churned. The classification follows a strict precedence rule to avoid overlap. A player is classified as new when first_seen_date equals snapshot_date. A player is classified as returning when active on the current day and previously classified as dormant or churned in the most recent prior observation. If neither condition applies, a player is classified as active when there is activity within the last 7 days. A player is classified as dormant when there has been no activity in the last 7 days but activity occurred within the last 30 days. A player is classified as churned when there has been no activity for more than 30 days.';
COMMENT ON COLUMN gold.characters_behavior_daily.lifecycle_event IS 'Player lifecycle transition event on the snapshot date. Mutually exclusive categories: new, returning, dormant, churned, or NULL when no transition occurs. The classification captures only meaningful state changes between consecutive observations. A player is classified as new when first_seen_date equals snapshot_date. A player is classified as returning when active on the current day and previously classified as dormant or churned in the most recent prior observation. A player is classified as dormant when transitioning from active to dormant, indicating loss of recent activity. A player is classified as churned when transitioning from dormant to churned, indicating long-term inactivity. If none of these transitions occur, the value is NULL.';
COMMENT ON COLUMN gold.characters_behavior_daily.in_guild IS 'Indicates whether the character is currently a member of a guild.';
COMMENT ON COLUMN gold.characters_behavior_daily.owns_house IS 'Indicates whether the character owns at least one house at the time of the snapshot.';
COMMENT ON COLUMN gold.characters_behavior_daily.is_married IS 'Indicates whether the character is currently married in the game.';
COMMENT ON COLUMN gold.characters_behavior_daily.loyalty_title IS 'The loyalty title associated with the character account.';
COMMENT ON COLUMN gold.characters_behavior_daily.achievement_points IS 'Total number of achievement points accumulated by the character at the time of the snapshot.';
COMMENT ON COLUMN gold.characters_behavior_daily.processed_at IS 'Timestamp when the record was processed into the Gold layer (UTC).';

In [0]:
CREATE TABLE IF NOT EXISTS gold.characters_behavior_periodic (
  granularity STRING,
  period_start DATE,
  period_end DATE,
  period_days INT,
  observed_days INT,
  is_partial_period BOOLEAN,
  period_status STRING,
  character_id STRING,
  world_id STRING,
  account_status STRING,
  sex STRING,
  vocation STRING,
  level_start INT,
  level INT,
  level_delta_period INT,
  level_delta_short INT,
  level_delta_medium INT,
  level_delta_long INT,
  level_delta_xlong INT,
  first_seen_date DATE,
  last_login_date DATE,
  days_since_last_login INT,
  is_active_in_period BOOLEAN,
  is_active_1d BOOLEAN,
  is_active_7d BOOLEAN,
  is_active_30d BOOLEAN,
  is_active_90d BOOLEAN,
  lifecycle_stage STRING,
  lifecycle_event STRING,
  new_events INT,
  returning_events INT,
  dormant_events INT,
  churned_events INT,
  in_guild BOOLEAN,
  owns_house BOOLEAN,
  is_married BOOLEAN,
  loyalty_title STRING,
  achievement_points INT,
  processed_at TIMESTAMP
);

In [0]:
COMMENT ON TABLE gold.characters_behavior_periodic IS 'Periodic behavioral snapshot of Tibia characters aggregated into standardized time windows for longitudinal community health, engagement, retention, and lifecycle analysis.';
COMMENT ON COLUMN gold.characters_behavior_periodic.granularity IS 'Aggregation level of the behavioral period. Supported values: Day, Week, Month, Quarter.';
COMMENT ON COLUMN gold.characters_behavior_periodic.period_start IS 'Start date of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.period_end IS 'End date of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.period_days IS 'Expected total number of calendar days within the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.observed_days IS 'Number of distinct snapshot dates effectively observed within the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_partial_period IS 'Indicates whether the aggregation period contains incomplete snapshot coverage.';
COMMENT ON COLUMN gold.characters_behavior_periodic.period_status IS 'Coverage classification of the aggregation period. Supported values: full, partial_start, partial_current, partial_missing. Full indicates complete expected coverage. Partial_start indicates the dataset collection started after the period beginning. Partial_current indicates the current ongoing period has not yet finished. Partial_missing indicates missing snapshots within the expected period range.';
COMMENT ON COLUMN gold.characters_behavior_periodic.character_id IS 'Stable unique identifier of the character.';
COMMENT ON COLUMN gold.characters_behavior_periodic.world_id IS 'Stable unique identifier of the Tibia world.';
COMMENT ON COLUMN gold.characters_behavior_periodic.account_status IS 'Indicates whether the character account is Free or Premium.';
COMMENT ON COLUMN gold.characters_behavior_periodic.sex IS 'Current sex of the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.vocation IS 'Current vocation of the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_start IS 'Character level observed at the beginning of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level IS 'Character level observed at the end of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_delta_period IS 'Difference between the character level at the end and the beginning of the aggregation period. Formula: level(period_end) - level(period_start).';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_delta_short IS 'Difference between the current character level and the level observed in the previous equivalent short-term comparison window. The comparison interval depends on granularity.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_delta_medium IS 'Difference between the current character level and the level observed in the previous equivalent medium-term comparison window. The comparison interval depends on granularity.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_delta_long IS 'Difference between the current character level and the level observed in the previous equivalent long-term comparison window. The comparison interval depends on granularity.';
COMMENT ON COLUMN gold.characters_behavior_periodic.level_delta_xlong IS 'Difference between the current character level and the level observed in the previous equivalent extra-long-term comparison window. The comparison interval depends on granularity.';
COMMENT ON COLUMN gold.characters_behavior_periodic.first_seen_date IS 'Earliest date on which the character was observed in the dataset based on ingestion history, not necessarily the character creation date.';
COMMENT ON COLUMN gold.characters_behavior_periodic.last_login_date IS 'Most recent known login date of the character as reported by the source API at the time of ingestion.';
COMMENT ON COLUMN gold.characters_behavior_periodic.days_since_last_login IS 'Number of days between the period_end and the character last_login_date, representing inactivity duration relative to the end of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_active_in_period IS 'Indicates whether the character logged in at least once during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_active_1d IS 'Indicates whether the character was active within the last 1 day relative to the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_active_7d IS 'Indicates whether the character was active within the last 7 days relative to the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_active_30d IS 'Indicates whether the character was active within the last 30 days relative to the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_active_90d IS 'Indicates whether the character was active within the last 90 days relative to the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.lifecycle_stage IS 'Player lifecycle classification at the end of the aggregation period. Mutually exclusive categories: new, returning, active, dormant, churned.';
COMMENT ON COLUMN gold.characters_behavior_periodic.lifecycle_event IS 'Highest-priority lifecycle transition event observed during the aggregation period. Mutually exclusive categories: new, returning, dormant, churned, or NULL when no transition occurs.';
COMMENT ON COLUMN gold.characters_behavior_periodic.new_events IS 'Number of new lifecycle transition events observed for the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.returning_events IS 'Number of returning lifecycle transition events observed for the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.dormant_events IS 'Number of dormant lifecycle transition events observed for the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.churned_events IS 'Number of churned lifecycle transition events observed for the character during the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.in_guild IS 'Indicates whether the character was a member of a guild in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.owns_house IS 'Indicates whether the character owned at least one house in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.is_married IS 'Indicates whether the character was married in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.loyalty_title IS 'The loyalty title associated with the character account in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.achievement_points IS 'Total number of achievement points accumulated by the character in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN gold.characters_behavior_periodic.processed_at IS 'Timestamp when the record was processed into the Gold layer (UTC).';
